# 03 — Simulação de Previsões

Este notebook utiliza o modelo salvo na etapa de modelagem para estimar o índice de perda em novos registros de materiais.

A ideia é representar uma situação prática: a partir de informações conhecidas antes da execução do serviço, o modelo estima o índice de perda esperado. Com esse índice, também é possível calcular uma quantidade recomendada de compra e uma estimativa de custo da perda.

In [ ]:
import pandas as pd
import joblib
from pathlib import Path

In [ ]:
# Carregando o modelo treinado
project_path = Path("..")
models_path = project_path / "models"
modelo_regressao = joblib.load(models_path / "modelo_perda_material.pkl")

In [ ]:
# Recuperando modelo e informações auxiliares
modelo = modelo_regressao["modelo"]
features = modelo_regressao["features"]
alvo = modelo_regressao["alvo"]
nome_modelo = modelo_regressao["nome_modelo"]

print("Modelo carregado:", nome_modelo)
print("Alvo:", alvo)
print("Quantidade de features:", len(features))

Modelo carregado: Regressão Linear
Alvo: indice_perda_real
Quantidade de features: 15


## 1. Previsão para um novo registro

Nesta etapa, é montado um novo registro de material com as mesmas variáveis usadas no treinamento.

O objetivo é aplicar o modelo salvo e obter o índice de perda previsto para esse caso.

In [ ]:
# Criando um novo registro com as mesmas variáveis usadas no treinamento
novo_registro = pd.DataFrame([{
    "tipo_obra": "comercial",
    "fase_obra": "acabamento",
    "ambiente": "interno",
    "servico": "revestimento_piso_parede",
    "complexidade_execucao": "media",
    "necessidade_corte_ajuste": "alta",
    "material": "porcelanato",
    "tipo_item_material": "principal",
    "material_fragil": 1,
    "material_modular": 1,
    "custo_unitario": 95.00,
    "quantidade_teorica": 300.00,
    "indice_perda_orcado": 0.08,
    "pressao_prazo": "media",
    "qtd_frentes_simultaneas": 3
}])

# Organizando o novo registro na mesma ordem de features usada no treinamento
novo_registro = novo_registro[features]

In [ ]:
# Gerando a previsão do índice de perda para o novo registro
indice_perda_previsto = modelo.predict(novo_registro)[0]

## 2. Conversão da previsão em indicadores de compra

Com o índice de perda previsto, calculo a quantidade ajustada para compra, a perda esperada e o custo estimado dessa perda.

Esses valores são comparados com o cálculo baseado no índice de perda orçado, permitindo avaliar a diferença prática entre a referência inicial e a previsão do modelo.

In [ ]:
# Calculando os principais resultados a partir da perda prevista
material = novo_registro["material"].iloc[0]
servico = novo_registro["servico"].iloc[0]

quantidade_teorica = novo_registro["quantidade_teorica"].iloc[0]
custo_unitario = novo_registro["custo_unitario"].iloc[0]
indice_perda_orcado = novo_registro["indice_perda_orcado"].iloc[0]

quantidade_orcada = quantidade_teorica * (1 + indice_perda_orcado)
quantidade_recomendada = quantidade_teorica * (1 + indice_perda_previsto)

quantidade_perda_orcada = quantidade_orcada - quantidade_teorica
quantidade_perda_prevista = quantidade_recomendada - quantidade_teorica

custo_perda_orcado = quantidade_perda_orcada * custo_unitario
custo_perda_previsto = quantidade_perda_prevista * custo_unitario

diferenca_quantidade = quantidade_recomendada - quantidade_orcada
diferenca_custo = custo_perda_previsto - custo_perda_orcado

In [ ]:
# Consolidando a saída da simulação
print("SIMULAÇÃO DE PREVISÃO DE PERDA DE MATERIAL")
print("-" * 50)

print(f"Material: {material}")
print(f"Serviço: {servico}")
print(f"Modelo utilizado: {nome_modelo}")

print("-" * 50)

print(f"Quantidade teórica: {quantidade_teorica:.2f}")
print(f"Custo unitário: R$ {custo_unitario:.2f}")

print("-" * 50)

print(f"Índice de perda orçado: {indice_perda_orcado:.2%}")
print(f"Índice de perda previsto pelo modelo: {indice_perda_previsto:.2%}")

print("-" * 50)

print(f"Quantidade orçada: {quantidade_orcada:.2f}")
print(f"Quantidade recomendada pelo modelo: {quantidade_recomendada:.2f}")

print("-" * 50)

print(f"Quantidade de perda orçada: {quantidade_perda_orcada:.2f}")
print(f"Quantidade de perda prevista: {quantidade_perda_prevista:.2f}")

print("-" * 50)

print(f"Custo da perda orçada: R$ {custo_perda_orcado:.2f}")
print(f"Custo da perda prevista: R$ {custo_perda_previsto:.2f}")

print("-" * 50)

print(f"Diferença de quantidade em relação ao orçamento: {diferenca_quantidade:.2f}")
print(f"Diferença de custo em relação ao orçamento: R$ {diferenca_custo:.2f}")

SIMULAÇÃO DE PREVISÃO DE PERDA DE MATERIAL
--------------------------------------------------
Material: porcelanato
Serviço: revestimento_piso_parede
Modelo utilizado: Regressão Linear
--------------------------------------------------
Quantidade teórica: 300.00
Custo unitário: R$ 95.00
--------------------------------------------------
Índice de perda orçado: 8.00%
Índice de perda previsto pelo modelo: 14.01%
--------------------------------------------------
Quantidade orçada: 324.00
Quantidade recomendada pelo modelo: 342.02
--------------------------------------------------
Quantidade de perda orçada: 24.00
Quantidade de perda prevista: 42.02
--------------------------------------------------
Custo da perda orçada: R$ 2280.00
Custo da perda prevista: R$ 3991.90
--------------------------------------------------
Diferença de quantidade em relação ao orçamento: 18.02
Diferença de custo em relação ao orçamento: R$ 1711.90


O modelo salvo foi carregado e aplicado a um novo registro de material.

A simulação mostra como o índice de perda previsto pode ser convertido em quantidade recomendada de compra e custo estimado da perda. Também permite comparar a previsão do modelo com o cálculo baseado no índice orçado.